In [1]:
import transformers
print(transformers.__version__)

5.0.0


In [2]:
import torch
import pickle
import json
from tabulate import tabulate
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score

In [3]:
data = json.loads("""
{
  "intents": [
    {
      "tag": "greeting",
      "patterns": [
        "hello",
        "hi",
        "hey",
        "good morning",
        "good evening",
        "hello there",
        "hi bookgenie",
        "hey assistant",
        "anyone there",
        "can you help me",
        "i need help",
        "support needed",
        "hello support",
        "hi support",
        "hey support",
        "greetings",
        "good afternoon",
        "hi there",
        "hello book genie",
        "hey book genie",
        "is anyone online",
        "can i get help",
        "i want help",
        "need assistance",
        "hello bot",
        "hi bot",
        "hey bot",
        "start chat",
        "help me please",
        "can you assist",
        "what's up",
        "yo",
        "namaste",
        "howdy",
        "hiya",
        "good day",
        "are you a robot",
        "is this a bot",
        "talk to me",
        "i have a question",
        "can i ask something",
        "help required",
        "assist me",
        "greetings of the day",
        "hey man",
        "hi buddy",
        "hello friend"
      ],
      "responses": [
        "Hello! Welcome to BookGenie. How can I help you today?",
        "Hi there! Need help finding a book?",
        "Welcome to BookGenie! How may I assist you?",
        "Hello! You can search, order, or get recommendations.",
        "Hi! What would you like to do today?",
        "Greetings! I am BookGenie, your personal bookstore assistant. What are you looking for?",
        "Good day! How can I make your reading journey better today?",
        "Hey! Ready to discover some amazing books? Ask me anything.",
        "Namaste! Welcome to our store. How can I assist you with your book shopping?",
        "Hello! Let me know if you need help tracking an order, finding a book, or checking your account."
      ]
    },
    {
      "tag": "goodbye",
      "patterns": [
        "bye",
        "goodbye",
        "see you",
        "exit",
        "quit",
        "close chat",
        "talk later",
        "bye bot",
        "goodbye assistant",
        "end chat",
        "stop chat",
        "see you later",
        "bye bookgenie",
        "thanks bye",
        "i am done",
        "nothing else",
        "that's all",
        "finish chat",
        "close conversation",
        "leave chat",
        "catch you later",
        "gotta go",
        "i'm leaving",
        "cya",
        "bye for now",
        "have a good day",
        "have a great day",
        "goodnight",
        "take care",
        "adios",
        "hasta la vista",
        "signing off",
        "i am signing off",
        "close the window"
      ],
      "responses": [
        "Goodbye! Have a great day.",
        "See you again soon!",
        "Thanks for visiting BookGenie!",
        "Bye! Happy reading!",
        "Take care! Hope to see you back for more books.",
        "Have a wonderful day ahead! Keep reading.",
        "Goodbye! Don't hesitate to return if you need more book recommendations.",
        "Signing off! Happy reading and take care.",
        "Catch you later! Your next great read is waiting for you here.",
        "Thanks for stopping by. Have a fantastic day!"
      ]
    },
    {
      "tag": "login_help",
      "patterns": [
        "cannot login",
        "login not working",
        "forgot password",
        "unable to login",
        "login error",
        "cannot sign in",
        "signin problem",
        "password not working",
        "account login issue",
        "login failed",
        "help me login",
        "how to login",
        "login help",
        "sign in help",
        "login support",
        "cannot access account",
        "login problem",
        "sign in failed",
        "i forgot my password",
        "reset password",
        "change password",
        "invalid credentials",
        "locked out of account",
        "account locked",
        "email not recognized",
        "login button not working",
        "where do i log in",
        "help with my account",
        "password reset link"
      ],
      "responses": [
        "Please ensure you are using the correct email and password.",
        "You can reset your password using the 'Forgot Password' link on the login page.",
        "If you are locked out, please try resetting your password or waiting 15 minutes.",
        "Make sure there are no extra spaces in your email or password.",
        "Click on the 'Profile' icon at the top right to access the login page.",
        "If you forgot your password, enter your email in the 'Forgot Password' section to get a reset link.",
        "Still having trouble? I can connect you to a support agent to unlock your account.",
        "Try clearing your browser cache and cookies, then attempt to log in again.",
        "Check your spam folder if you requested a password reset link but haven't received it.",
        "Your account might be temporarily locked due to multiple failed attempts. Try again later."
      ]
    },
    {
      "tag": "signup_help",
      "patterns": [
        "how to signup",
        "create account",
        "register account",
        "signup help",
        "cannot signup",
        "signup problem",
        "registration help",
        "new account",
        "open account",
        "create new account",
        "register help",
        "signup issue",
        "account registration",
        "sign up error",
        "cannot register",
        "join now",
        "become a member",
        "how to register",
        "signup page not working",
        "where to create account",
        "i want to sign up",
        "make an account"
      ],
      "responses": [
        "To sign up, click on the 'Register' button at the top right corner of our website.",
        "Creating an account is free! Just provide your name, email, and a secure password.",
        "Having trouble registering? Ensure your email format is correct and passwords match.",
        "Once you sign up, you'll receive a confirmation email to verify your account.",
        "You can also sign up quickly using your Google or Facebook account.",
        "If it says the email is already in use, try logging in or resetting your password.",
        "Registration only takes a minute. Click 'Sign Up' and follow the on-screen instructions.",
        "Please ensure your password meets the security requirements (at least 8 characters, 1 number).",
        "As a new member, you'll get a 10% discount on your first purchase after signing up!",
        "If the signup form is not loading, please try refreshing the page or using a different browser."
      ]
    },
    {
      "tag": "search_book",
      "patterns": [
        "find book",
        "search book",
        "find harry potter",
        "search python book",
        "book search",
        "i want a book",
        "find novel",
        "search by author",
        "search by title",
        "look for book",
        "show books",
        "find programming book",
        "search data science book",
        "find ai book",
        "search machine learning book",
        "search by isbn",
        "find isbn",
        "do you have this book",
        "looking for a specific book",
        "help me find a book",
        "search title",
        "find this author",
        "is this book available",
        "check book availability",
        "where is the search bar"
      ],
      "responses": [
        "You can use the search bar at the top of the page to find books by title, author, or ISBN.",
        "Please type the name of the book or the author you are looking for.",
        "Looking for something specific? Enter the ISBN in the search bar for an exact match.",
        "You can browse through our categories or use the search feature to find your next read.",
        "Tell me the title of the book, and I'll see if we have it in stock.",
        "We have thousands of books! Use the advanced search filters to narrow down by genre, price, or format.",
        "If you know the author's name, just search for it to see all their available titles.",
        "Can't find it? Make sure the spelling is correct, or try searching with fewer keywords.",
        "Our search bar supports title, author, publisher, and keyword searches.",
        "To search, simply click the magnifying glass icon and type your query."
      ]
    },
    {
      "tag": "book_details",
      "patterns": [
        "book details",
        "show book info",
        "show book details",
        "book information",
        "details of book",
        "tell me about this book",
        "book description",
        "book author",
        "book publisher",
        "book price",
        "how many pages",
        "what is the isbn",
        "who published this",
        "read synopsis",
        "book summary",
        "what is this book about",
        "is it hardcover or paperback",
        "book dimensions",
        "publication date",
        "when was this published",
        "who wrote this"
      ],
      "responses": [
        "You can find the full synopsis, author details, and publication info on the book's product page.",
        "Click on any book cover to view its detailed description, page count, and ISBN.",
        "The product details section includes the format (Hardcover/Paperback), publisher, and release date.",
        "If you want to know what a book is about, check the 'Summary' section on its page.",
        "Book dimensions, weight, and language are listed under 'Product Specifications'.",
        "To see who wrote the book and read their biography, scroll down to the 'About the Author' section.",
        "Pricing, available formats, and customer reviews are all available on the book's specific page.",
        "We list the exact publication date and edition under the book details.",
        "Want to know the page count? It's mentioned right below the book description.",
        "If you are on the book page, you can see all technical details like ISBN-10 and ISBN-13."
      ]
    },
    {
      "tag": "add_to_cart",
      "patterns": [
        "add to cart",
        "add book to cart",
        "put in cart",
        "add this book",
        "add item",
        "cart add",
        "add this",
        "add product",
        "add book",
        "how to buy this",
        "i want to purchase this",
        "put this in my basket",
        "add to basket",
        "save to cart",
        "buy item",
        "purchase book",
        "how do i add items",
        "where is the add button"
      ],
      "responses": [
        "To add a book, simply click the 'Add to Cart' button on the product page.",
        "Once you find a book you like, hit 'Add to Cart' to place it in your shopping basket.",
        "Item successfully added to your cart! You can continue shopping or proceed to checkout.",
        "You can add multiple quantities by adjusting the number before clicking 'Add to Cart'.",
        "Added! A popup will confirm that the item is now in your cart.",
        "If you're on the category page, you can use the quick 'Add' button below the book cover.",
        "Your book has been placed in the basket. You can view it by clicking the cart icon.",
        "Make sure to select your preferred format (eBook, Hardcover, etc.) before adding to the cart.",
        "To buy this, select 'Add to Cart'. You can always remove it later if you change your mind.",
        "Successfully added! Let me know if you want to go straight to checkout."
      ]
    },
    {
      "tag": "remove_from_cart",
      "patterns": [
        "remove from cart",
        "delete from cart",
        "remove item",
        "delete item",
        "remove book",
        "cart remove",
        "delete book",
        "remove product",
        "take it out of my basket",
        "empty cart",
        "clear cart",
        "delete everything in cart",
        "i don't want this book",
        "cancel item from cart",
        "remove this from my order"
      ],
      "responses": [
        "To remove an item, go to your cart and click the trash can icon next to the book.",
        "You can delete items from your cart by clicking the 'Remove' button under the quantity.",
        "If you want to clear your entire cart, look for the 'Empty Cart' button at the bottom.",
        "Item removed from your cart successfully. The total price has been updated.",
        "To take a book out of your basket, open the cart and hit the 'X' next to the product.",
        "Changed your mind? No problem. Go to the cart page to manage and remove items.",
        "Your cart has been updated and the selected item has been deleted.",
        "You can easily remove products or change the quantity to zero in your shopping cart.",
        "If you accidentally added something, just open your cart and click 'Remove'.",
        "Done! The item has been deleted from your shopping basket."
      ]
    },
    {
      "tag": "view_cart",
      "patterns": [
        "show cart",
        "view cart",
        "open cart",
        "my cart",
        "cart items",
        "cart details",
        "show my cart",
        "see cart",
        "what is in my cart",
        "check basket",
        "view basket",
        "how many items in my cart",
        "cart total",
        "show me my basket",
        "take me to cart"
      ],
      "responses": [
        "You can view your cart by clicking the shopping bag icon at the top right of the screen.",
        "Here is your cart! You can review your items before proceeding to checkout.",
        "Taking you to your shopping cart now.",
        "To see what's in your basket, click on the cart icon in the navigation bar.",
        "Your cart details, including total price and shipping estimates, are available on the Cart page.",
        "Opening your cart. Let me know if you need help modifying your items.",
        "You currently have items in your cart. Click the cart icon to view the full list.",
        "You can check your subtotal and applied discounts by viewing your cart.",
        "Redirecting you to the cart page so you can see your selected books.",
        "Your cart is ready! Review your items and proceed to checkout when you're done."
      ]
    },
    {
      "tag": "checkout",
      "patterns": [
        "how to checkout",
        "proceed to checkout",
        "buy now",
        "place order",
        "checkout cart",
        "complete purchase",
        "pay for books",
        "finish shopping",
        "how do i pay",
        "go to payment",
        "where to checkout",
        "checkout process",
        "confirm order",
        "make payment",
        "ready to buy"
      ],
      "responses": [
        "Go to your cart and click the 'Proceed to Checkout' button to complete your purchase.",
        "Ready to buy? Just hit the checkout button in your cart and follow the steps.",
        "During checkout, you will need to provide your shipping address and payment details.",
        "To place your order, view your cart and select 'Checkout'.",
        "You can checkout as a guest or log in to your account for faster processing.",
        "Click 'Buy Now' to skip the cart and go straight to the payment page.",
        "Once you are in the checkout area, review your order, select shipping, and pay.",
        "To finish your shopping, head to the cart icon and click 'Proceed to Checkout'.",
        "Ensure all your details are correct on the checkout page before confirming the order.",
        "Checkout is secure and fast. Just click the checkout button to finalize your book order."
      ]
    },
    {
      "tag": "payment_methods",
      "patterns": [
        "how can i pay",
        "payment methods",
        "do you accept paypal",
        "credit card",
        "debit card",
        "payment options",
        "how to pay",
        "cash on delivery",
        "is cod available",
        "apple pay",
        "google pay",
        "upi payment",
        "can i pay later",
        "payment gateways",
        "which cards are accepted",
        "net banking",
        "emi options"
      ],
      "responses": [
        "We accept Visa, Mastercard, American Express, PayPal, and major digital wallets.",
        "You can pay securely using credit/debit cards, Net Banking, or UPI.",
        "Yes, Cash on Delivery (COD) is available for select pin codes. You can check eligibility at checkout.",
        "We support Apple Pay and Google Pay for faster mobile checkouts.",
        "All our payment gateways are 100% secure and encrypted.",
        "If you are buying premium books, we also offer EMI options through select credit cards.",
        "You can choose your preferred payment method on the final checkout page.",
        "We accept direct bank transfers for wholesale or bulk corporate orders.",
        "Unfortunately, we do not accept personal checks or money orders at this time.",
        "Yes, we accept PayPal! Just select the PayPal option during checkout."
      ]
    },
    {
      "tag": "shipping_delivery",
      "patterns": [
        "how long does shipping take",
        "delivery time",
        "shipping cost",
        "when will my book arrive",
        "delivery tracking",
        "shipping methods",
        "shipping fee",
        "free shipping",
        "how much for delivery",
        "expedited shipping",
        "same day delivery",
        "courier service",
        "delivery options",
        "how many days to deliver",
        "shipping charges"
      ],
      "responses": [
        "Standard shipping takes 3-5 business days depending on your location.",
        "Shipping costs are calculated at checkout based on the weight of the books and your address.",
        "We offer free standard shipping on all orders over $50!",
        "Need it faster? We offer Expedited (2-day) and Express (Overnight) shipping options.",
        "Delivery times vary by region, but most domestic orders arrive within a week.",
        "Once your order ships, you will receive an email with the courier tracking details.",
        "For same-day delivery, please check if your zip code is eligible during checkout.",
        "Shipping charges are clearly displayed in your cart before you make the payment.",
        "Our primary courier partners are FedEx, UPS, and local postal services.",
        "If you order an eBook or Audiobook, delivery is instant via digital download!"
      ]
    },
    {
      "tag": "return_refund",
      "patterns": [
        "can i return a book",
        "refund policy",
        "how to return",
        "get a refund",
        "return policy",
        "money back",
        "cancel order",
        "i want to return this",
        "exchange book",
        "return window",
        "how many days to return",
        "refund status",
        "where is my refund",
        "item exchange",
        "wrong book received"
      ],
      "responses": [
        "You can return any physical book in its original condition within 30 days of delivery.",
        "Refunds are processed to your original payment method within 5-7 business days after we receive the return.",
        "To initiate a return, go to your 'Order History', select the item, and click 'Return Item'.",
        "Please note that eBooks, audiobooks, and personalized items are non-refundable.",
        "If you received a damaged or incorrect book, please contact support immediately for a free replacement.",
        "Return shipping is free if the error was on our end. Otherwise, a small return fee may be deducted.",
        "Once your return is approved, we will email you a printable shipping label.",
        "You can track your refund status directly from your account dashboard.",
        "If you prefer an exchange instead of a refund, just select 'Exchange' when initiating the return.",
        "We have a hassle-free 30-day money-back guarantee for all physical books."
      ]
    },
    {
      "tag": "book_formats",
      "patterns": [
        "do you have ebooks",
        "kindle format",
        "audiobooks",
        "pdf books",
        "digital books",
        "listen to books",
        "is this an ebook",
        "hardcover",
        "paperback",
        "what formats do you sell",
        "epub format",
        "mp3 audiobook",
        "do you sell digital copies",
        "physical copy",
        "large print"
      ],
      "responses": [
        "We offer books in Hardcover, Paperback, eBook (ePub/MOBI), and Audiobook formats.",
        "You can choose your preferred format on the book's detail page before adding it to your cart.",
        "Our eBooks are DRM-free and compatible with most e-readers, including Kindle, iPad, and Kobo.",
        "Digital downloads (eBooks and Audiobooks) are available in your account immediately after purchase.",
        "If you prefer listening, look for the 'Audiobook' icon next to the book title.",
        "Yes, we sell physical books! You can filter your search results to show only Hardcovers or Paperbacks.",
        "Many of our popular titles are also available in Large Print for easier reading.",
        "Once you buy an eBook, you can read it directly on our mobile app or download the ePub file.",
        "Audiobooks can be streamed via our app or downloaded as MP3 files.",
        "Make sure to double-check the format you are buying in your cart to avoid getting a digital copy when you wanted a physical one!"
      ]
    },
    {
      "tag": "promotions_discounts",
      "patterns": [
        "discount codes",
        "promo code",
        "any offers",
        "coupons",
        "sale",
        "cheap books",
        "student discount",
        "apply coupon",
        "festival offers",
        "clearance sale",
        "how to get discount",
        "voucher code",
        "gift code",
        "is there a sale going on",
        "best deals"
      ],
      "responses": [
        "You can apply promo codes and discount vouchers on the checkout page.",
        "Check our 'Deals' page for current sales, clearance books, and daily discounts.",
        "Sign up for our newsletter to receive an exclusive 10% off coupon for your first order!",
        "Yes, we offer a 15% student discount! Verify your student ID through our partner link at the footer.",
        "During festive seasons, we run massive sales with up to 50% off on bestsellers.",
        "If you have a promo code, type it in the 'Apply Coupon' box in your cart and hit apply.",
        "Keep an eye on our homepage banner for the latest flash sales and promotional offers.",
        "Bundle offers are automatically applied. Buy 2 get 1 free on select fiction titles!",
        "Promo codes cannot be stacked. You can only use one code per order.",
        "If your promo code isn't working, check its expiration date or terms and conditions."
      ]
    },
    {
      "tag": "account_management",
      "patterns": [
        "update profile",
        "change email",
        "change address",
        "my account settings",
        "edit profile",
        "view order history",
        "my past orders",
        "delete account",
        "change phone number",
        "add new address",
        "manage addresses",
        "account info",
        "where is my profile",
        "update billing details",
        "change name"
      ],
      "responses": [
        "You can manage your shipping addresses and personal details in the 'My Account' section.",
        "To view past purchases and invoices, navigate to 'Order History' in your profile dashboard.",
        "Need to update your email or phone number? Click on 'Edit Profile' under your account settings.",
        "You can save multiple shipping addresses in your 'Address Book' for faster checkout.",
        "To update your billing details or saved cards, go to 'Payment Methods' in your account.",
        "If you wish to change your password, you can do so in the 'Security' tab of your profile.",
        "You can view, download, or print invoices for all your past orders from the Order History page.",
        "If you want to permanently delete your account, please contact customer support.",
        "Click your profile picture icon at the top right to access all account management tools.",
        "Your account dashboard allows you to track orders, manage wishlists, and update preferences."
      ]
    },
    {
      "tag": "order_status",
      "patterns": [
        "track order",
        "order status",
        "where is my order",
        "track my book",
        "order tracking",
        "check order",
        "track shipment",
        "has my order shipped",
        "when will it be delivered",
        "tracking number",
        "track package",
        "delivery status",
        "track awb",
        "courier tracking"
      ],
      "responses": [
        "You can track your order directly by visiting the 'Track Order' page and entering your Order ID.",
        "Order tracking is available on the 'My Orders' page in your account dashboard.",
        "Once your order ships, we send you an email with a tracking link and courier details.",
        "If your order status says 'Processing', it means we are currently packing your books.",
        "Status 'Shipped' means the package has been handed over to the courier partner.",
        "Enter your Order ID and Email Address on our tracking portal to see real-time updates.",
        "If your tracking link isn't updating, please wait 24 hours for the courier system to sync.",
        "You can also track your shipment directly on the courier partner's website using the AWB number provided.",
        "If your order is marked as 'Delivered' but you haven't received it, please check with neighbors or contact support.",
        "Guest users can track their orders using the link provided in their order confirmation email."
      ]
    },
    {
      "tag": "pre_orders",
      "patterns": [
        "how to pre order",
        "pre order book",
        "upcoming books",
        "reserve a book",
        "when is this coming out",
        "preorder release date",
        "can i book in advance",
        "pre-order details",
        "how does preorder work",
        "cancel preorder"
      ],
      "responses": [
        "You can pre-order upcoming books by clicking the 'Pre-Order' button on the book's page.",
        "When you pre-order, your card is charged immediately, and the book ships on the release date.",
        "Pre-ordering guarantees you get a copy from the first print run!",
        "You can find the expected release date listed under the book details for any pre-order item.",
        "If the price of a pre-order drops before release, we will automatically refund you the difference.",
        "You can cancel a pre-order anytime before it ships from your 'Order History' page.",
        "Check our 'Coming Soon' section to browse highly anticipated upcoming releases.",
        "Pre-ordered eBooks will automatically download to your library on the release date.",
        "If you order an in-stock item with a pre-order item, they will ship separately so you don't have to wait.",
        "Pre-orders are eligible for our standard return policy once you receive the book."
      ]
    },
    {
      "tag": "gift_cards",
      "patterns": [
        "buy gift card",
        "redeem gift card",
        "check gift card balance",
        "gift voucher",
        "how to use gift card",
        "send a gift card",
        "corporate gifting",
        "gift balance",
        "e-gift card",
        "physical gift card"
      ],
      "responses": [
        "We offer both physical and digital (e-Gift) cards! You can purchase them from our 'Gift Cards' page.",
        "To redeem a gift card, enter the card number and PIN in the 'Gift Card' section at checkout.",
        "You can check your remaining gift card balance in your 'Account Settings' under the 'Gift Cards' tab.",
        "e-Gift cards are emailed instantly to the recipient, making them the perfect last-minute gift.",
        "Gift cards do not expire and can be used on any item in the store, including sale items.",
        "If your order total is less than the gift card balance, the remaining amount will stay on the card for future use.",
        "You can apply multiple gift cards to a single order during checkout.",
        "Physical gift cards come in beautiful packaging and take 3-5 days for delivery.",
        "We also offer bulk gift cards for corporate gifting. Please contact our B2B sales team.",
        "Please treat your gift card like cash. We cannot replace lost or stolen gift card codes."
      ]
    },
    {
      "tag": "loyalty_program",
      "patterns": [
        "loyalty points",
        "reward points",
        "how to earn points",
        "membership program",
        "book club",
        "vip member",
        "redeem points",
        "what are reward points",
        "premium membership",
        "join loyalty program"
      ],
      "responses": [
        "Join our BookGenie Rewards program for free and earn points on every purchase!",
        "You earn 10 points for every $1 spent. 1000 points equal a $10 discount.",
        "To redeem your loyalty points, choose the 'Pay with Points' option at checkout.",
        "Members get early access to sales, exclusive discounts, and free shipping on orders over $25.",
        "You can view your current points balance in your Account Dashboard.",
        "Earn bonus points by leaving book reviews or referring friends to our store!",
        "Points expire after 12 months of account inactivity, so make sure to keep reading!",
        "Sign up for our Premium Book Club membership for flat 10% off all year round.",
        "Loyalty points cannot be earned on the purchase of gift cards.",
        "Welcome to the club! New members get 500 bonus points just for signing up."
      ]
    },
    {
      "tag": "bulk_orders",
      "patterns": [
        "bulk order",
        "buy in bulk",
        "wholesale books",
        "corporate order",
        "books for school",
        "large quantity",
        "bulk discount",
        "b2b orders",
        "how to buy 100 books",
        "school library order"
      ],
      "responses": [
        "We offer generous discounts for bulk orders (10+ copies of the same title).",
        "For corporate, school, or library orders, please fill out the form on our 'Bulk Orders' page.",
        "Bulk discounts start at 15% off and can go up to 40% depending on the quantity.",
        "We can assist educators in sourcing textbooks and literature sets for classrooms.",
        "Please allow extra processing time for large wholesale orders.",
        "Contact our B2B support team at corporate@bookgenie.com for a custom quote.",
        "Yes, we can provide proforma invoices and accept wire transfers for bulk purchases.",
        "Shipping for bulk orders is usually handled via freight forwarding to ensure safe delivery.",
        "We supply books for corporate gifting, events, and organizational libraries.",
        "Let us know the ISBNs and quantities you need, and our bulk sales team will assist you."
      ]
    },
    {
      "tag": "used_books",
      "patterns": [
        "used books",
        "second hand books",
        "old books",
        "buy used",
        "condition of used books",
        "sell my books",
        "do you buy used books",
        "cheap used books",
        "pre-loved books",
        "thrift books"
      ],
      "responses": [
        "Yes, we have a massive collection of pre-loved and used books at great prices!",
        "You can filter your search results to show only 'Used' condition books.",
        "Our used books are graded: Like New, Very Good, Good, and Acceptable.",
        "We inspect all used books to ensure there are no missing pages or severe damage.",
        "Looking to declutter? We buy back used books! Check our 'Sell to Us' page for details.",
        "Used books are a great way to save money and read sustainably.",
        "Please note that used books might have minor wear, dog-eared pages, or previous owner notes.",
        "Stock for used books is limited, so grab them before they are gone!",
        "Used books are generally shipped within the same timeframe as new books.",
        "If you receive a used book that doesn't match its condition grading, we will issue a full refund."
      ]
    },
    {
      "tag": "store_pickup",
      "patterns": [
        "store pickup",
        "click and collect",
        "pick up in store",
        "bopis",
        "curbside pickup",
        "local store",
        "is this in my local store",
        "where is your physical store",
        "collect from store"
      ],
      "responses": [
        "We offer free 'Buy Online, Pick Up In-Store' (BOPIS) at all our retail locations.",
        "During checkout, select 'Store Pickup' and choose the nearest store to your zip code.",
        "You will receive an email notification as soon as your order is ready for pickup.",
        "Most in-stock items are ready for pickup within 2 hours of placing the order.",
        "Please bring your order confirmation email and a valid photo ID when you come to collect.",
        "We also offer contactless Curbside Pickup! Just park in the designated spot and call the store.",
        "If you don't pick up your order within 7 days, it will be cancelled and refunded.",
        "You can check if a specific book is in stock at your local store on the product page.",
        "To find a store near you, visit our 'Store Locator' page.",
        "Store pickup saves you shipping fees and gets the book in your hands faster!"
      ]
    },
    {
      "tag": "genre_scifi_fantasy",
      "patterns": [
        "sci fi books",
        "fantasy books",
        "recommend sci fi",
        "science fiction",
        "dragons",
        "space opera",
        "magic books",
        "cyberpunk",
        "best fantasy",
        "dystopian novels"
      ],
      "responses": [
        "Our Sci-Fi & Fantasy section has everything from epic high fantasy to futuristic space operas.",
        "Looking for dragons and magic? Check out our Top 10 Fantasy Bestsellers list!",
        "If you love Sci-Fi, I highly recommend browsing our Hugo and Nebula Award winners.",
        "We have huge collections for fans of Brandon Sanderson, George R.R. Martin, and Isaac Asimov.",
        "Check out the 'Dystopian & Cyberpunk' category for gritty futuristic reads.",
        "You can browse by sub-genres like Urban Fantasy, Hard Sci-Fi, and Steampunk.",
        "Ready to start a new series? We offer box sets for popular fantasy trilogies.",
        "Our trending Sci-Fi books are currently listed on the homepage under 'Out of this World'.",
        "Whether you want aliens or elves, our Sci-Fi & Fantasy page has it all.",
        "Filter your search by 'Science Fiction' to explore galaxies far, far away."
      ]
    },
    {
      "tag": "genre_mystery_thriller",
      "patterns": [
        "mystery books",
        "thriller novels",
        "crime fiction",
        "detective books",
        "suspense",
        "murder mystery",
        "psychological thriller",
        "recommend mystery",
        "whodunit",
        "scary books"
      ],
      "responses": [
        "Love a good plot twist? Browse our Mystery, Thriller & Suspense category.",
        "We have a massive collection of Crime Fiction and Detective Novels.",
        "Check out our 'Psychological Thrillers' section for books that will keep you up all night.",
        "Looking for a whodunit? Agatha Christie and Arthur Conan Doyle are always great choices.",
        "Our best-selling thrillers are updated weekly on the 'Gripping Reads' page.",
        "Filter your search to find cozy mysteries, police procedurals, or espionage thrillers.",
        "If you like true crime, we have a dedicated Non-Fiction Crime section as well.",
        "Get your adrenaline pumping with our top-rated action and suspense novels.",
        "We highly recommend the latest domestic thrillers for a fast-paced read.",
        "Explore the darker side of fiction in our Mystery & Crime department."
      ]
    },
    {
      "tag": "genre_romance",
      "patterns": [
        "romance novels",
        "love stories",
        "recommend romance",
        "romantic comedy",
        "historical romance",
        "contemporary romance",
        "spicy books",
        "romantasy",
        "enemies to lovers",
        "booktok romance"
      ],
      "responses": [
        "Fall in love with our huge selection of Romance novels!",
        "We have all your favorite tropes: Enemies to Lovers, Fake Dating, and Grumpy x Sunshine.",
        "Looking for Historical Romance? We have plenty of Regency and Victorian era love stories.",
        "Check out our 'BookTok Favorites' for the most viral and trending romance books.",
        "Browse the 'Romantasy' section for the perfect blend of romance and fantasy.",
        "Our Contemporary Romance section features the latest bestsellers.",
        "If you're looking for something spicy, check out our New Adult romance category.",
        "We have a great selection of heartwarming Romantic Comedies.",
        "Filter by 'Romance' to find your next happily ever after.",
        "Looking for diverse romance? We have dedicated sections for LGBTQ+ love stories."
      ]
    },
    {
      "tag": "genre_nonfiction_biography",
      "patterns": [
        "biography",
        "autobiography",
        "memoirs",
        "non fiction",
        "history books",
        "self help",
        "business books",
        "true story",
        "personal development",
        "recommend non fiction"
      ],
      "responses": [
        "Our Non-Fiction section includes History, Science, Business, and Self-Help.",
        "Explore fascinating lives in our Biographies & Memoirs category.",
        "Looking to improve yourself? Browse our top-rated Personal Development books.",
        "We have a comprehensive selection of Business & Economics literature.",
        "History buffs will love our extensive World History and Military History sections.",
        "If you want to read a true story, check out our award-winning Memoirs.",
        "Browse our Science & Nature category to learn something new about the world.",
        "We carry all the latest bestsellers in Politics and Social Sciences.",
        "Filter by 'Non-Fiction' to dive into facts, philosophy, and real-world events.",
        "Our Self-Help section covers everything from productivity to mental health."
      ]
    },
    {
      "tag": "genre_manga_comics",
      "patterns": [
        "manga",
        "comics",
        "graphic novels",
        "anime books",
        "marvel comics",
        "dc comics",
        "japanese manga",
        "webtoons",
        "buy manga",
        "comic books"
      ],
      "responses": [
        "Welcome to the Manga & Comics section! We have everything from Shonen to Seinen.",
        "We carry major publishers like VIZ Media, Kodansha, Marvel, DC, and Image Comics.",
        "Looking for a specific Manga volume? Use the search bar to find exact issues.",
        "Our Graphic Novels section includes critically acclaimed standalone stories and series.",
        "We stock the latest Box Sets for popular Manga series.",
        "Fans of Webtoons will find printed editions of their favorite online comics here.",
        "Check out our 'New Releases' in the comic section for the latest Wednesday drops.",
        "We offer comic book trade paperbacks, omnibuses, and absolute editions.",
        "Filter your search by 'Manga' or 'Graphic Novel' to find visually stunning stories.",
        "Whether you love superheroes or slice-of-life anime, we have the perfect comic for you."
      ]
    },
    {
      "tag": "international_shipping",
      "patterns": [
        "ship internationally",
        "outside country",
        "international delivery",
        "ship to uk",
        "ship to india",
        "global shipping",
        "do you deliver abroad",
        "customs fee",
        "overseas shipping",
        "international rates"
      ],
      "responses": [
        "Yes, we ship books globally to over 150 countries!",
        "International shipping rates are calculated at checkout based on destination and package weight.",
        "Delivery times for international orders usually range from 10 to 21 business days.",
        "Please note that international customers are responsible for any customs duties or import taxes.",
        "We offer DHL Express for fast, trackable international delivery.",
        "To check if we ship to your country, simply select your destination in the cart shipping calculator.",
        "Unfortunately, we cannot ship heavy Box Sets internationally due to exorbitant freight costs.",
        "International orders are fully tracked from our warehouse to your doorstep.",
        "If your country is not listed at checkout, please contact support for a custom shipping quote.",
        "Digital items like eBooks and Audiobooks are available worldwide with no shipping fees!"
      ]
    },
    {
      "tag": "out_of_stock",
      "patterns": [
        "out of stock",
        "when will this be back",
        "sold out",
        "book not available",
        "restock date",
        "notify me when back in stock",
        "currently unavailable",
        "can i backorder",
        "waiting list"
      ],
      "responses": [
        "If a book is out of stock, you can click 'Notify Me' on the product page to get an email alert when it returns.",
        "We usually restock popular titles within 1-2 weeks.",
        "Some out-of-stock items allow 'Backordering', meaning you can buy it now and it will ship the moment it arrives.",
        "If a book is marked 'Out of Print', we unfortunately won't be receiving new copies.",
        "You might want to check if the eBook or Audiobook version is available immediately!",
        "Our inventory updates daily. Keep an eye on the product page for changes.",
        "If you need a sold-out book urgently, try checking our 'Used Books' section.",
        "We are currently waiting on a shipment from the publisher for this title.",
        "Clicking the 'Notify Me' button is the best way to secure a copy from the next batch.",
        "If a specific edition is out of stock, try searching to see if we have a different edition (e.g., Paperback instead of Hardcover)."
      ]
    },
    {
      "tag": "contact_support",
      "patterns": [
        "contact support",
        "talk to human",
        "customer support",
        "connect to agent",
        "support help",
        "live agent",
        "human support",
        "speak to representative",
        "call customer service",
        "phone number",
        "email address",
        "help desk",
        "i have a complaint",
        "escalate issue"
      ],
      "responses": [
        "Our support team is here for you. I am connecting you to a live human agent now. Please hold.",
        "You can reach our customer service via email at support@bookgenie.com or call 1-800-BOOK-HLP.",
        "Transferring you to a live support representative...",
        "Our live agents are available Monday to Friday, 9 AM to 6 PM EST.",
        "If your issue is urgent, please call our toll-free support line.",
        "I understand you need more help. Let me open a support ticket for you.",
        "You can chat with a human agent by clicking the 'Agent' button at the bottom of the chat.",
        "To help the agent assist you faster, please have your Order ID ready.",
        "Our support desk usually responds to emails within 24 hours.",
        "I'm sorry I couldn't resolve this. Let me get a human to help you out immediately."
      ]
    },
    {
      "tag": "thanks",
      "patterns": [
        "thanks",
        "thank you",
        "thanks a lot",
        "thank you bot",
        "appreciate it",
        "thanks assistant",
        "thank you support",
        "great help",
        "awesome thanks",
        "much appreciated",
        "you are helpful",
        "good bot",
        "thanks for the info",
        "perfect thanks"
      ],
      "responses": [
        "You're very welcome! Let me know if you need anything else.",
        "Happy to help! Enjoy your reading.",
        "Anytime! I'm always here to assist you.",
        "Glad I could help. Have a wonderful day!",
        "You're welcome! Don't hesitate to ask if you have more questions.",
        "It's my pleasure! Finding the right book is what I do best.",
        "Thanks for choosing BookGenie! Happy reading!",
        "I appreciate your kind words! Is there anything else I can do for you?",
        "Awesome! Let me know when you're ready for your next book recommendation.",
        "You are welcome! Have a great time exploring our bookstore."
      ]
    },
    {
      "tag": "fallback",
      "patterns": [
        "asdfasdf",
        "random text",
        "blah blah",
        "unknown",
        "not sure",
        "something random",
        "invalid input",
        "gibberish",
        "i don't know",
        "what?",
        "huh?",
        "can you repeat that",
        "xxyyzz"
      ],
      "responses": [
        "I'm sorry, I didn't quite catch that. Could you rephrase your question?",
        "I'm not sure I understand. Try asking about searching for books, checking your order, or managing your account.",
        "Could you explain that a bit differently? I'm just a bookstore bot!",
        "I don't have the answer to that right now. Want me to connect you to a live support agent?",
        "Hmm, I didn't get that. Try asking me to 'Find a book' or 'Track my order'.",
        "Oops! I didn't understand. Can you please type that again?",
        "My apologies, I am trained to help with bookstore queries. Could you ask me about books, shipping, or payments?",
        "I'm still learning! If you need specific help, you can say 'Contact Support'.",
        "Sorry, that didn't make sense to me. Check out our FAQ page for common questions!",
        "I missed that. Please try using simpler keywords like 'Return policy' or 'Search author'."
      ]
    }
  ]
}
""")

In [4]:
texts = []
labels = []

for intent in data["intents"]:
    for pattern in intent["patterns"]:
        texts.append(pattern)
        labels.append(intent["tag"])

le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels_encoded, test_size=0.2, random_state=42
)

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
class IntentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IntentDataset(train_encodings, train_labels)
val_dataset = IntentDataset(val_encodings, val_labels)

In [6]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(le.classes_)
)

training_args = TrainingArguments(
    output_dir="./intent_model_results",
    num_train_epochs=40,
    per_device_train_batch_size=8,
    logging_dir="./logs",
    learning_rate=3e-5
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [7]:
def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, preds)}

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [9]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,1.562665
1000,0.042626
1500,0.008507
2000,0.005437


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2080, training_loss=0.3894321096344636, metrics={'train_runtime': 1241.745, 'train_samples_per_second': 13.272, 'train_steps_per_second': 1.675, 'total_flos': 38395364351040.0, 'train_loss': 0.3894321096344636, 'epoch': 40.0})

In [13]:
model.save_pretrained("./intent_model")
tokenizer.save_pretrained("./intent_model")

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Training Done!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Done!


In [16]:
import torch
import random
import pickle
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

# Load model
model = DistilBertForSequenceClassification.from_pretrained("./intent_model")
tokenizer = DistilBertTokenizerFast.from_pretrained("./intent_model")

with open("label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

model.eval()

# Response function
def predict_intent(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs).item()
    return le.inverse_transform([pred])[0]

def get_response(tag):
    for intent in data["intents"]:
        if intent["tag"] == tag:
            return random.choice(intent["responses"])
    return "Sorry, samajh nahi aaya"
print("Bot ready! (type 'quit' to exit)")

while True:
    user = input("You: ")

    if user.lower() == "quit":
        break

    tag = predict_intent(user)
    response = get_response(tag)

    print("Bot:", response)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Bot ready! (type 'quit' to exit)
You: hey
Bot: Hi there! Need help finding a book?
You: how are you
Bot: Hello! Welcome to BookGenie. How can I help you today?
You: can u help me in finding a book
Bot: Tell me the title of the book, and I'll see if we have it in stock.
You: sasaram
Bot: Hello! You can search, order, or get recommendations.
You: i am suraj kumar
Bot: Hello! You can search, order, or get recommendations.
You: quit
